# HHS Healthcare Breach Data: Cleaning

This notebook turns two raw exports from the [HHS Office for Civil Rights breach portal](https://ocrportal.hhs.gov/ocr/breach/breach_report.jsf) into one clean table that's ready for analysis. The portal lists breaches of protected health information that affected 500 or more people, in two lists:

- **Under investigation**: cases the Office for Civil Rights (OCR) is still investigating.
- **Archive**: older and resolved cases, going back to 2009.

Each step below explains what it does and why. No analysis happens here. The result, `data/clean/breaches_clean.csv`, is the starting point for the analysis notebooks.

## Load the raw exports

Both CSV files are read exactly as downloaded. Each one's size, column names and first three rows are printed, so there's a record of the starting point. If a file isn't valid UTF-8, the loader tries again with `latin-1` encoding. Both current exports load without needing that.

Note the two odd column names that start with `javax.faces.component.UIPanel@`. Step 1 deals with them.

In [1]:
import re
import textwrap
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")


def load_csv(path):
    """Load a CSV with pandas' default encoding (UTF-8), retrying with latin-1 if that fails."""
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        print(f"{path.name}: default encoding failed, retrying with latin-1")
        return pd.read_csv(path, encoding="latin-1")


under_investigation = load_csv(RAW / "under_investigation.csv")
archive = load_csv(RAW / "archive.csv")

for name, df in [("under_investigation.csv", under_investigation), ("archive.csv", archive)]:
    print(f"===== {name} =====")
    print("Shape:", df.shape)
    print("Columns:")
    for col in df.columns:
        print("   ", col)
    print("First 3 rows (transposed, one line per column):")
    with pd.option_context("display.width", 250, "display.max_colwidth", 40):
        print(df.head(3).T)
    print()

===== under_investigation.csv =====
Shape: (754, 9)
Columns:
    javax.faces.component.UIPanel@12254ae0
    State
    Covered Entity Type
    Individuals Affected
    Breach Submission Date
    Type of Breach
    Location of Breached Information
    javax.faces.component.UIPanel@30bd38a3
    Web Description
First 3 rows (transposed, one line per column):
                                                          0                          1                                        2
javax.faces.component.UIPanel@12254ae0    Woodlawn Hospital  Imagine the Possibilities  Applied Healthcare Nursing Division,...
State                                                    IN                         IA                                       TX
Covered Entity Type                     Healthcare Provider                Health Plan                      Healthcare Provider
Individuals Affected                                    742                       1693                                    14333
Bre

## Step 1: Fix and standardize the column names

The portal's export has a bug. Two headers come out as internal web-framework names (`javax.faces.component.UIPanel@…`) instead of *Name of Covered Entity* and *Business Associate Present*. The code after the `@` also changes with every download, so these columns can't be renamed by name. Instead they're renamed by **position**: the 1st column becomes `Entity Name` and the 8th becomes `Business Associate Present`.

Every column name is then converted to `snake_case` (lowercase, words joined by underscores), so the names are easy to type and the same in both files. A check stops the notebook if the portal ever reorders its columns, because renaming by position would then put the wrong labels on the data.

In [2]:
# 1st and 8th columns (Python counts from 0)
POSITIONAL_RENAMES = {0: "Entity Name", 7: "Business Associate Present"}

EXPECTED_COLUMNS = [
    "entity_name",
    "state",
    "covered_entity_type",
    "individuals_affected",
    "breach_submission_date",
    "type_of_breach",
    "location_of_breached_information",
    "business_associate_present",
    "web_description",
]


def to_snake_case(text):
    """'Breach Submission Date' -> 'breach_submission_date', 'Paper/Films' -> 'paper_films'"""
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def standardize_columns(df):
    names = list(df.columns)
    for position, name in POSITIONAL_RENAMES.items():
        names[position] = name
    df = df.set_axis([to_snake_case(name) for name in names], axis=1)
    assert list(df.columns) == EXPECTED_COLUMNS, f"Unexpected column layout: {list(df.columns)}"
    return df


broken_headers = [col for col in [*under_investigation.columns, *archive.columns] if col.startswith("javax.faces")]
under_investigation = standardize_columns(under_investigation)
archive = standardize_columns(archive)

print(f"Replaced {len(broken_headers)} broken headers: {broken_headers}")
print("Columns in both files now:", list(under_investigation.columns))

Replaced 4 broken headers: ['javax.faces.component.UIPanel@12254ae0', 'javax.faces.component.UIPanel@30bd38a3', 'javax.faces.component.UIPanel@145d9745', 'javax.faces.component.UIPanel@104fbd55']
Columns in both files now: ['entity_name', 'state', 'covered_entity_type', 'individuals_affected', 'breach_submission_date', 'type_of_breach', 'location_of_breached_information', 'business_associate_present', 'web_description']


## Step 2: Label each record's status and combine the files

Which list a record comes from shows whether OCR is still investigating it, and stacking the files would lose that. So each file first gets a `status` column (`Under Investigation` or `Archived`), and then the two are combined into one table.

The under-investigation records go on top, because step 3 uses that order to decide which copy of a duplicate to keep.

`web_description` is empty for every under-investigation record, so pandas read that column as numbers in that file. It's converted to text before combining, so the combined column has one consistent text type.

In [3]:
under_investigation["status"] = "Under Investigation"
archive["status"] = "Archived"

# Blank in every under-investigation row, so pandas read it as numbers; make it text like the archive's
under_investigation["web_description"] = under_investigation["web_description"].astype("str")

# Under-investigation rows first: step 3 relies on this order
breaches = pd.concat([under_investigation, archive], ignore_index=True)
rows_before = len(breaches)

print(breaches["status"].value_counts().to_string())
print(f"Combined: {rows_before:,} rows x {breaches.shape[1]} columns")

status
Archived               7261
Under Investigation     754
Combined: 8,015 rows x 10 columns


## Step 3: Check for duplicates

Duplicate records would inflate breach counts, so two checks run before anything is removed:

1. **Exact duplicates** are rows that match in every original column. The `status` column is ignored, so a report that appears in both lists also counts.
2. **The same breach in both lists** means the same organization and submission date appear in both files, even if other details differ. A case can move from one list to the other between downloads. In that case the under-investigation copy is kept, because it's the more cautious label: it doesn't assume the case is closed.

Organization names are compared after trimming spaces and ignoring capitalization, so `"Acme Health "` and `"ACME HEALTH"` count as the same name.

Rows that share an organization and date *within* one file are reported but kept. They usually report different numbers of people, which points to separate reports, such as a vendor filing one report for each client. Rows that also share the same number of people are listed as possible repeat filings for manual review.

In [4]:
SHOW_COLUMNS = ["entity_name", "state", "individuals_affected", "breach_submission_date", "type_of_breach", "status"]
is_open = breaches["status"].eq("Under Investigation")

# Names compared trimmed, single-spaced and lowercased, so spacing or capitalization can't hide a match
name_key = breaches["entity_name"].str.strip().str.replace(r"\s+", " ", regex=True).str.casefold()
match_key = name_key + " | " + breaches["breach_submission_date"].str.strip()

# 1. Exact duplicates across every original column. keep="first" keeps the under-investigation copy.
exact_duplicate = breaches.duplicated(subset=EXPECTED_COLUMNS, keep="first")
in_exact_group = breaches.duplicated(subset=EXPECTED_COLUMNS, keep=False)

# 2. Same organization + submission date in both files: the archive copy goes
keys_in_both_files = set(match_key[is_open]) & set(match_key[~is_open])
archived_copy_of_open_case = ~is_open & match_key.isin(keys_in_both_files)

# For context, kept: the same organization + date more than once within one file (not exact copies)
keyed = breaches.assign(match_key=match_key)
shared_key = keyed.duplicated(subset=["status", "match_key"], keep=False) & ~in_exact_group
same_people_too = keyed.duplicated(subset=["status", "match_key", "individuals_affected"], keep=False) & ~in_exact_group
shared_groups = keyed[shared_key].groupby(["status", "match_key"])["individuals_affected"].nunique()


def strip_text(column):
    """Trimmed copy of a text column, so whitespace-only differences don't count as differences."""
    return column.str.strip() if pd.api.types.is_string_dtype(column) else column


print("Found before removing anything:")
print(f"  Exact duplicate rows (extra copies):       {exact_duplicate.sum():,}")
print(f"  Same organization + date in both files:    {len(keys_in_both_files):,}")
print(f"  Same organization + date within one file:  {shared_key.sum():,} rows in {len(shared_groups):,} groups, "
      f"{(shared_groups > 1).mean():.0%} of them with different numbers of people affected")

print("\nExact duplicates (all copies):")
print(breaches.loc[in_exact_group, SHOW_COLUMNS].to_string())

print("\nSame organization, date and number of people, but not identical (possible repeat filings, kept):")
possible_repeat_filings = []
for _, group in keyed[same_people_too].groupby(["status", "match_key", "individuals_affected"]):
    first = group.iloc[0]
    differs = [col for col in EXPECTED_COLUMNS if strip_text(group[col]).nunique(dropna=False) > 1]
    possible_repeat_filings.append(first["entity_name"].strip())
    print(f"  {first['entity_name'].strip()} ({first['breach_submission_date']}, {first['individuals_affected']:,.0f} people): "
          f"{len(group)} rows that differ in {', '.join(differs)}")

exact_duplicate_names = breaches.loc[exact_duplicate, "entity_name"].str.strip().value_counts()
to_drop = exact_duplicate | archived_copy_of_open_case
removed = breaches[to_drop]
breaches = breaches[~to_drop].reset_index(drop=True)
print(f"\nRemoved {len(removed):,} rows ({exact_duplicate.sum():,} exact copies + "
      f"{(archived_copy_of_open_case & ~exact_duplicate).sum():,} archive copies of open cases): "
      f"{rows_before:,} -> {len(breaches):,}")

Found before removing anything:
  Exact duplicate rows (extra copies):       2
  Same organization + date in both files:    0
  Same organization + date within one file:  117 rows in 44 groups, 91% of them with different numbers of people affected

Exact duplicates (all copies):
             entity_name state  individuals_affected breach_submission_date       type_of_breach               status
526  McEwen & Associates    TX                 500.0             08/21/2025  Hacking/IT Incident  Under Investigation
527  McEwen & Associates    TX                 500.0             08/21/2025  Hacking/IT Incident  Under Investigation
528  McEwen & Associates    TX                 500.0             08/21/2025  Hacking/IT Incident  Under Investigation

Same organization, date and number of people, but not identical (possible repeat filings, kept):
  Area Agency of Aging 1-B (04/13/2017, 1,741 people): 2 rows that differ in web_description
  Original Medicine Acupuncture & Wellness, LLC (11/21/20

## Step 4: Clean the columns

### 4a. People affected → whole numbers

`individuals_affected` counts people, so it should hold whole numbers. A single blank value is enough to make pandas store a whole column as decimals, which is why the archive shows values like `840.0`. Converting to pandas' nullable integer type, `Int64`, gives whole numbers again. The blank stays a proper missing value (`<NA>`) instead of becoming a misleading 0.

In [5]:
breaches["individuals_affected"] = breaches["individuals_affected"].astype("Int64")

missing_people = breaches["individuals_affected"].isna()
print(f"individuals_affected is now {breaches['individuals_affected'].dtype}, with {missing_people.sum()} blank value(s):")
print(breaches.loc[missing_people, ["entity_name", "state", "breach_submission_date", "status"]].to_string())

individuals_affected is now Int64, with 1 blank value(s):
                     entity_name state breach_submission_date    status
7212  Valperaiso Fire Department    IN             09/03/2013  Archived


### 4b. Submission date → real dates, plus year, quarter and month

Dates arrive as US-style text (`09/21/2026`). As text they sort in the wrong order (`01/05/2024` comes before `12/31/2009`) and can't be grouped by time period. They're parsed with an explicit `MM/DD/YYYY` format, which stops with an error on any value that doesn't fit instead of quietly guessing. Three helper columns make trend charts easier later on: `year`, `quarter` (1 to 4) and `year_month` (for example `2026-09`).

In [6]:
dates = pd.to_datetime(breaches["breach_submission_date"], format="%m/%d/%Y")

breaches["breach_submission_date"] = dates
breaches["year"] = dates.dt.year.astype("Int64")
breaches["quarter"] = dates.dt.quarter.astype("Int64")
breaches["year_month"] = dates.dt.to_period("M")

print(f"Parsed {dates.notna().sum():,} dates ({dates.isna().sum()} blank), from {dates.min():%Y-%m-%d} to {dates.max():%Y-%m-%d}")
print(breaches[["breach_submission_date", "year", "quarter", "year_month"]].head(3).to_string())

Parsed 8,013 dates (0 blank), from 2009-10-21 to 2026-09-21
  breach_submission_date  year  quarter year_month
0             2026-09-21  2026        3    2026-09
1             2026-09-21  2026        3    2026-09
2             2026-09-19  2026        3    2026-09


### 4c. Business associate present → True/False

This column records whether a business associate was involved in the breach. A business associate is an outside company handling health data for the organization, such as a billing service or an IT provider. The raw values are the text `Yes` and `No`. As True/False the column can be counted and filtered directly. The conversion first checks for unexpected values such as `Y` or `Unknown`, so nothing is silently turned into a blank.

In [7]:
YES_NO = {"Yes": True, "No": False}

unexpected = set(breaches["business_associate_present"].dropna()) - set(YES_NO)
assert not unexpected, f"Unexpected values in business_associate_present: {unexpected}"

breaches["business_associate_present"] = breaches["business_associate_present"].map(YES_NO).astype("boolean")
print(breaches["business_associate_present"].value_counts(dropna=False).to_string())

business_associate_present
False    5615
True     2398


### 4d. Trim whitespace and review category spellings

Hundreds of organization names have invisible extra spaces at the start or end (`"Acme Clinic "`). A computer treats those as different names from the same text without the spaces. Leading and trailing spaces are removed from every text column, and runs of spaces inside organization names are reduced to one.

Then every distinct value of the four category columns is listed alphabetically with its count, so misspellings and inconsistent labels (for example `Health Plan` vs `Health plan`) appear next to each other. In this download all four are consistent: states are standard two-letter codes, and each breach type and location is always spelled the same way.

In [8]:
TEXT_COLUMNS = [col for col in breaches.columns if pd.api.types.is_string_dtype(breaches[col])]
CATEGORY_COLUMNS = ["state", "covered_entity_type", "type_of_breach", "location_of_breached_information"]

values_changed = {}
for col in TEXT_COLUMNS:
    cleaned = breaches[col].str.strip()
    if col == "entity_name":
        cleaned = cleaned.str.replace(r"\s+", " ", regex=True)  # "Acme  Health" -> "Acme Health"
    values_changed[col] = int((cleaned.ne(breaches[col]) & breaches[col].notna()).sum())
    breaches[col] = cleaned

print("Values changed by whitespace cleanup:", {col: n for col, n in values_changed.items() if n} or "none")

for col in CATEGORY_COLUMNS:
    counts = breaches[col].value_counts(dropna=False).sort_index()
    print(f"\n{col}: {breaches[col].nunique()} distinct values, {breaches[col].isna().sum()} blank")
    for value, n in counts.items():
        print(f"  {n:>6,}  {'<blank>' if pd.isna(value) else value}")

Values changed by whitespace cleanup: {'entity_name': 747, 'web_description': 1379}

state: 52 distinct values, 23 blank
      32  AK
     100  AL
      94  AR
     167  AZ
     786  CA
     141  CO
     146  CT
      34  DC
      32  DE
     471  FL
     229  GA
      24  HI
     100  IA
      32  ID
     364  IL
     203  IN
      83  KS
     124  KY
      67  LA
     245  MA
     172  MD
      39  ME
     241  MI
     194  MN
     166  MO
      55  MS
      36  MT
     208  NC
      21  ND
      60  NE
      41  NH
     177  NJ
      57  NM
      55  NV
     516  NY
     278  OH
      83  OK
     120  OR
     345  PA
      32  PR
      41  RI
      87  SC
      15  SD
     200  TN
     654  TX
      68  UT
     158  VA
      16  VT
     187  WA
     126  WI
      49  WV
      19  WY
      23  <blank>

covered_entity_type: 4 distinct values, 5 blank
   1,229  Business Associate
     969  Health Plan
      17  Healthcare Clearing House
   5,793  Healthcare Provider
       5  <blank>



## Step 5: Split multi-value columns into True/False flags

One breach can involve several breach types (`"Hacking/IT Incident, Theft"`) and several locations (`"Email, Network Server"`). In that text form, counting is awkward, because `"Email"` and `"Email, Network Server"` look like different categories. The original columns are kept for reference, and one True/False column is added per distinct value: `type_…` for breach types and `loc_…` for locations. A breach can be True in several of them, so each set of counts adds up to more than the number of breaches.

In [9]:
MULTI_VALUE_COLUMNS = {"type_of_breach": "type", "location_of_breached_information": "loc"}
SHORT_NAMES = {"Hacking/IT Incident": "hacking", "Unauthorized Access/Disclosure": "unauthorized_access"}

flag_columns = {}
for col, prefix in MULTI_VALUE_COLUMNS.items():
    # "Email, Network Server" -> loc_email = True and loc_network_server = True; a blank gives all False
    flags = breaches[col].str.replace(r"\s*,\s*", ",", regex=True).str.get_dummies(sep=",").astype(bool)
    flags.columns = [f"{prefix}_{SHORT_NAMES.get(value, to_snake_case(value))}" for value in flags.columns]
    breaches = breaches.join(flags)
    flag_columns[col] = list(flags.columns)

    several = (flags.sum(axis=1) > 1).sum()
    print(f"{col}: {len(flags.columns)} flag columns; {several:,} breaches have more than one value")
    print(flags.sum().sort_values(ascending=False).to_string(), end="\n\n")

type_of_breach: 7 flag columns; 86 breaches have more than one value
type_hacking                4830
type_unauthorized_access    1741
type_theft                  1060
type_loss                    247
type_improper_disposal       127
type_other                    95
type_unknown                  16

location_of_breached_information: 8 flag columns; 520 breaches have more than one value
loc_network_server                      3691
loc_email                               1855
loc_paper_films                         1012
loc_laptop                               514
loc_electronic_medical_record            477
loc_other                                462
loc_desktop_computer                     380
loc_other_portable_electronic_device     331



## Step 6: Flag mega-breaches

Breach sizes are extremely lopsided. A typical report involves a few thousand people, but a handful involve tens of millions, and those few records dominate any total or average. A `mega_breach` column (True when 1,000,000 or more people were affected) makes it easy to show results with and without these outliers later. Records with an unknown count are marked False, because they can't be confirmed as mega-breaches.

In [10]:
MEGA_BREACH_THRESHOLD = 1_000_000

# Unknown counts (<NA>) can't be confirmed as mega-breaches, so they become False
breaches["mega_breach"] = breaches["individuals_affected"].ge(MEGA_BREACH_THRESHOLD).fillna(False).astype(bool)
print(f"Mega-breaches (1,000,000+ people): {breaches['mega_breach'].sum():,} of {len(breaches):,} records")

largest = breaches.sort_values("individuals_affected", ascending=False).head(10)
print("\nThe 10 largest breaches:")
print(largest[["entity_name", "year", "individuals_affected"]].to_string(index=False, formatters={"individuals_affected": "{:,}".format}))

Mega-breaches (1,000,000+ people): 152 of 8,013 records

The 10 largest breaches:
                        entity_name  year individuals_affected
            Change Healthcare, Inc.  2024          192,700,000
                        Anthem Inc.  2015           78,800,000
     Conduent Business Services LLC  2025           62,224,658
                    DentaQuest, LLC  2026           15,000,000
                      Welltok, Inc.  2023           14,782,887
       Aflac Incorporated (“Aflac”)  2025           13,924,906
Kaiser Foundation Health Plan, Inc.  2024           13,400,000
                      Optum360, LLC  2019           11,500,000
                     HCA Healthcare  2023           11,270,000
                 Premera Blue Cross  2015           11,000,000


## Step 7: Save the clean table

The result is saved to `data/clean/breaches_clean.csv`. That folder is excluded from Git, because rerunning this notebook on the raw exports rebuilds the file, so a fresh copy of the repository won't have it yet. The notebook creates the folder if it's missing.

CSV files don't store column types, so notebooks that load this file need to parse `breach_submission_date` as a date again.

In [11]:
CLEAN_DIR = Path("../data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

clean_path = CLEAN_DIR / "breaches_clean.csv"
breaches.to_csv(clean_path, index=False)
print(f"Saved {len(breaches):,} rows x {breaches.shape[1]} columns to {clean_path}")

Saved 8,013 rows x 29 columns to ../data/clean/breaches_clean.csv


## Step 8: Data quality summary

This is a single place that records what the cleaning changed and what to watch for during analysis: row counts before and after, duplicates removed, remaining blanks, the date range covered, and quirks found along the way. Every number is calculated from the data, so the summary stays accurate if the notebook is rerun on a newer download.

In [12]:
open_cases = breaches["status"].eq("Under Investigation")
dates = breaches["breach_submission_date"]
people = breaches["individuals_affected"]
added_columns = [col for col in breaches.columns if col not in EXPECTED_COLUMNS]
blanks = breaches[EXPECTED_COLUMNS].isna().sum()
other_blanks = blanks.drop("web_description")
open_without_description = breaches.loc[open_cases, "web_description"].isna().sum()
spellings_per_name = breaches.groupby(breaches["entity_name"].str.casefold())["entity_name"].nunique()
case_variants = spellings_per_name[spellings_per_name > 1].index
case_example = breaches.loc[breaches["entity_name"].str.casefold().isin(case_variants[:1]), "entity_name"].unique()
several_types = breaches[flag_columns["type_of_breach"]].sum(axis=1).gt(1).sum()
several_locations = breaches[flag_columns["location_of_breached_information"]].sum(axis=1).gt(1).sum()

print("DATA QUALITY SUMMARY")
print("=" * 72)
print(f"Rows before cleaning:  {rows_before:,}  ({len(under_investigation):,} under investigation + {len(archive):,} archived)")
print(f"Duplicates removed:    {len(removed):,}  ({exact_duplicate.sum():,} exact copies, "
      f"{archived_copy_of_open_case.sum():,} archive copies of open cases)")
print(f"Rows after cleaning:   {len(breaches):,}  ({open_cases.sum():,} under investigation + {(~open_cases).sum():,} archived)")
print(f"Columns:               {breaches.shape[1]}  ({len(EXPECTED_COLUMNS)} original + {len(added_columns)} added)")
print(f"Date range covered:    {dates.min():%Y-%m-%d} to {dates.max():%Y-%m-%d}")

print(f"\nBlank values per column (the {len(added_columns)} added columns have {breaches[added_columns].isna().sum().sum()}):")
print(pd.DataFrame({"blank": blanks, "% of rows": (blanks / len(breaches) * 100).round(1)}).to_string())

notes = [
    f"Broken export headers: {len(broken_headers)} column names came out as 'javax.faces.component.UIPanel@...' and were "
    "renamed by position in step 1. Check the column order whenever a new export is downloaded.",

    "Duplicates: removed "
    + (", ".join(f"{n} extra cop{'y' if n == 1 else 'ies'} of {name}" for name, n in exact_duplicate_names.items())
       or "no exact copies")
    + ". The export has no report ID, so identical rows could in principle be separate filings. "
    + (f"{len(keys_in_both_files)} breaches appeared in both files; their archive copies were dropped."
       if keys_in_both_files else "No breach appeared in both files, so the keep-the-open-case rule removed nothing."),

    f"{shared_key.sum():,} rows ({len(shared_groups)} groups) share an organization and submission date with another row in "
    f"the same file. {(shared_groups > 1).mean():.0%} of these groups report different numbers of people, which suggests "
    "separate reports (for example, a vendor filing one per client), so they were kept. Groups that also match on the number "
    f"of people may be repeat filings worth a manual check: {', '.join(possible_repeat_filings) or 'none'} (step 3 shows what differs).",

    f"{len(case_variants)} organization names appear with different capitalization"
    + (f" (e.g. {' vs '.join(case_example)})" if len(case_example) else "")
    + ". They were left as written, so lowercase names before counting breaches per organization.",

    "Blanks outside web_description are rare and were left blank rather than guessed: "
    + ", ".join(f"{col} {n}" for col, n in other_blanks[other_blanks > 0].items()) + ".",

    f"web_description is blank for {open_without_description:,} of {open_cases.sum():,} open cases and for "
    f"{blanks['web_description'] - open_without_description:,} archived ones, so descriptions only exist for archived cases.",

    f"The flag columns overlap: {several_types:,} breaches list more than one type and {several_locations:,} more than one "
    "location, so type or location counts add up to more than the number of breaches.",

    f"Partial years: reports start {dates.min():%Y-%m-%d} and end {dates.max():%Y-%m-%d}, so {dates.min().year} and "
    f"{dates.max().year} are incomplete. Compare full years only.",

    f"Status is not the same as age: open cases go back to {dates[open_cases].min():%Y-%m-%d}, while the archive includes "
    f"cases from as recently as {dates[~open_cases].max():%Y-%m-%d}.",

    f"Breach sizes are extremely skewed: the median breach affected {people.median():,.0f} people but the mean is "
    f"{people.mean():,.0f}. The {breaches['mega_breach'].sum():,} mega-breaches dominate any total or average.",

    f"Sanity checks: the smallest breach affected {people.min():,} people (the portal's threshold is 500), "
    f"{dates.isna().sum()} dates failed to parse, and {breaches.duplicated(subset=EXPECTED_COLUMNS).sum()} exact "
    "duplicates remain after whitespace cleanup.",
]

print("\nThings to keep in mind:")
for number, note in enumerate(notes, start=1):
    print(textwrap.fill(note, width=100, initial_indent=f"{number:>2}. ", subsequent_indent="    "))

DATA QUALITY SUMMARY
Rows before cleaning:  8,015  (754 under investigation + 7,261 archived)
Duplicates removed:    2  (2 exact copies, 0 archive copies of open cases)
Rows after cleaning:   8,013  (752 under investigation + 7,261 archived)
Columns:               29  (9 original + 20 added)
Date range covered:    2009-10-21 to 2026-09-21

Blank values per column (the 20 added columns have 0):
                                  blank  % of rows
entity_name                           0        0.0
state                                23        0.3
covered_entity_type                   5        0.1
individuals_affected                  1        0.0
breach_submission_date                0        0.0
type_of_breach                        1        0.0
location_of_breached_information      0        0.0
business_associate_present            0        0.0
web_description                     946       11.8

Things to keep in mind:
 1. Broken export headers: 4 column names came out as 'javax.faces.c